# Mobilidade Urbana Inteligente: reconstrução e redistribuição da frota

**Global Solution 2026.2 · CitizenTech · Modelagem Matemática e Computacional**
Lucas Barreto Santana · RM 573149 · 1CCPG

Este notebook resolve as duas etapas do enunciado com **sistemas lineares, matrizes e transformações lineares**:

1. **Etapa 1:** recuperar quantos ônibus cada região tem hoje, a partir dos indicadores do Relatório I.
2. **Etapa 2:** comparar com a demanda do Relatório II e propor uma redistribuição, escrita como transformação linear.

Os cálculos são feitos de duas formas: com um **escalonamento implementado à mão** em frações exatas (`fractions.Fraction`, sem erro de arredondamento) e com o **NumPy**, como conferência.

Ligação com o projeto Ponto Vivo: o abrigo inteligente conta quantas pessoas esperam em cada ponto. Esses dados podem substituir a pesquisa de pico do Relatório II e alimentar este mesmo modelo com a demanda real.

In [1]:
from fractions import Fraction
import numpy as np

REGIOES = ["Norte", "Sul", "Leste", "Oeste"]

def fracoes(matriz):
    # Converte uma matriz (lista de listas) para frações exatas
    return [[Fraction(valor) for valor in linha] for linha in matriz]

def mostrar_matriz(matriz, nome=""):
    # Imprime a matriz alinhada, com frações quando houver
    textos = [[str(valor) for valor in linha] for linha in matriz]
    largura = max(len(texto) for linha in textos for texto in linha)
    if nome:
        print(nome)
    for linha in textos:
        print("  [ " + "  ".join(texto.rjust(largura) for texto in linha) + " ]")

---
## Etapa 1: diagnóstico e modelagem da situação atual

### a. Variáveis

| Variável | Significado | Domínio |
|---|---|---|
| $N$ | ônibus alocados no centro operacional **Norte** | inteiro, $N \ge 0$ |
| $S$ | ônibus alocados no centro operacional **Sul** | inteiro, $S \ge 0$ |
| $L$ | ônibus alocados no centro operacional **Leste** | inteiro, $L \ge 0$ |
| $O$ | ônibus alocados no centro operacional **Oeste** | inteiro, $O \ge 0$ |

A distribuição da frota é o vetor $x = (N, S, L, O)^T$.

### b. Modelo matemático

Cada indicador preservado do Relatório I vira uma equação linear:

| Indicador do Relatório I | Equação |
|---|---|
| A frota disponível é de 72 ônibus | $N + S + L + O = 72$ |
| Norte + Oeste + o dobro do Sul = 76 | $N + 2S + O = 76$ |
| Sul + o dobro do Leste = 52 | $S + 2L = 52$ |
| Norte + Leste + o dobro do Oeste = 76 | $N + L + 2O = 76$ |

São **4 equações e 4 incógnitas**, com as restrições de que a solução seja inteira e não negativa.

### c. Sistema matricial

$$
\underbrace{\begin{bmatrix} 1 & 1 & 1 & 1 \\ 1 & 2 & 0 & 1 \\ 0 & 1 & 2 & 0 \\ 1 & 0 & 1 & 2 \end{bmatrix}}_{A}
\underbrace{\begin{bmatrix} N \\ S \\ L \\ O \end{bmatrix}}_{x}
=
\underbrace{\begin{bmatrix} 72 \\ 76 \\ 52 \\ 76 \end{bmatrix}}_{b}
$$

Cada **linha** de $A$ é um indicador do relatório e cada **coluna** é uma região. O coeficiente $a_{ij}$ diz quantas vezes os ônibus da região $j$ entram no indicador $i$ (0 = não entram, 1 = entram uma vez, 2 = entram em dobro).

In [2]:
A = fracoes([
    [1, 1, 1, 1],   # frota total
    [1, 2, 0, 1],   # N + 2S + O
    [0, 1, 2, 0],   # S + 2L
    [1, 0, 1, 2],   # N + L + 2O
])
b = [Fraction(v) for v in (72, 76, 52, 76)]

mostrar_matriz(A, "A =")
print("b =", [int(v) for v in b])

A =
  [ 1  1  1  1 ]
  [ 1  2  0  1 ]
  [ 0  1  2  0 ]
  [ 1  0  1  2 ]
b = [72, 76, 52, 76]


### d. A distribuição é única?

Pelo **Teorema de Rouché–Capelli**, um sistema $Ax = b$ com $n$ incógnitas:

- é **possível e determinado** (uma única solução) quando $\text{posto}(A) = \text{posto}([A \mid b]) = n$;
- é **possível e indeterminado** (infinitas soluções) quando os postos são iguais, mas menores que $n$;
- é **impossível** quando $\text{posto}(A) < \text{posto}([A \mid b])$.

Para uma matriz quadrada, $\text{posto}(A) = n$ é o mesmo que $\det(A) \neq 0$, ou seja, $A$ é **inversível** e a única solução é $x = A^{-1}b$.

Abaixo, o escalonamento (eliminação de Gauss) da matriz aumentada $[A \mid b]$ é feito passo a passo. Ele dá ao mesmo tempo o **posto** (número de pivôs) e o **determinante** (produto dos pivôs, trocando o sinal a cada troca de linhas).

In [3]:
def escalonar(matriz_aumentada, mostrar_passos=True):
    # Eliminação de Gauss com frações exatas.
    # Devolve a matriz escalonada, o posto da parte dos coeficientes, o posto da
    # matriz aumentada e o determinante (só faz sentido se a parte A for quadrada).
    M = [linha[:] for linha in matriz_aumentada]
    linhas, colunas = len(M), len(M[0])
    n = colunas - 1                      # a última coluna é o vetor b
    determinante = Fraction(1)
    linha_pivo = 0
    for coluna in range(n):
        # procura uma linha com valor diferente de zero nesta coluna
        escolhida = next((i for i in range(linha_pivo, linhas) if M[i][coluna] != 0), None)
        if escolhida is None:
            determinante = Fraction(0)   # coluna sem pivô: det(A) = 0
            continue
        if escolhida != linha_pivo:
            M[linha_pivo], M[escolhida] = M[escolhida], M[linha_pivo]
            determinante = -determinante
            if mostrar_passos:
                print(f"Troca L{linha_pivo + 1} <-> L{escolhida + 1}")
        pivo = M[linha_pivo][coluna]
        determinante *= pivo
        for i in range(linha_pivo + 1, linhas):
            fator = M[i][coluna] / pivo
            if fator != 0:
                M[i] = [M[i][k] - fator * M[linha_pivo][k] for k in range(colunas)]
                if mostrar_passos:
                    print(f"L{i + 1} <- L{i + 1} - ({fator}) * L{linha_pivo + 1}")
        linha_pivo += 1
    posto_A = sum(1 for linha in M if any(valor != 0 for valor in linha[:n]))
    posto_Ab = sum(1 for linha in M if any(valor != 0 for valor in linha))
    return M, posto_A, posto_Ab, determinante


aumentada = [linha + [valor] for linha, valor in zip(A, b)]
escalonada, posto_A, posto_Ab, det_A = escalonar(aumentada)

print()
mostrar_matriz(escalonada, "[A | b] escalonada =")
print()
print(f"posto(A)       = {posto_A}")
print(f"posto([A | b]) = {posto_Ab}")
print(f"incógnitas     = {len(A)}")
print(f"det(A)         = {det_A}")
print()
print("Conferência com NumPy:")
print("  det(A)   =", round(np.linalg.det(np.array(A, dtype=float)), 10))
print("  posto(A) =", np.linalg.matrix_rank(np.array(A, dtype=float)))

L2 <- L2 - (1) * L1
L4 <- L4 - (1) * L1
L3 <- L3 - (1) * L2
L4 <- L4 - (-1) * L2
L4 <- L4 - (-1/3) * L3

[A | b] escalonada =
  [  1   1   1   1  72 ]
  [  0   1  -1   0   4 ]
  [  0   0   3   0  48 ]
  [  0   0   0   1  24 ]

posto(A)       = 4
posto([A | b]) = 4
incógnitas     = 4
det(A)         = 3

Conferência com NumPy:
  det(A)   = 3.0
  posto(A) = 4


**Conclusão do item d:** o escalonamento encontrou **4 pivôs**, então $\text{posto}(A) = \text{posto}([A \mid b]) = 4 = n$, e $\det(A) = 3 \neq 0$. Pelo Teorema de Rouché–Capelli, o sistema é **possível e determinado**: as informações do Relatório I permitem recuperar **uma única** distribuição de ônibus. Como $\det(A) \neq 0$, as quatro equações são **linearmente independentes**: nenhum indicador pode ser obtido combinando os outros, então nenhum deles é redundante.

Isso só garante uma solução real única. Ainda é preciso conferir (item f) se ela é **inteira e não negativa**, como exige o enunciado; se não fosse, os registros seriam inconsistentes com uma frota real.

### e. Resolução em Python

A solução sai da matriz escalonada por **substituição regressiva**: resolve-se a última equação (uma incógnita), depois a penúltima, e assim por diante.

In [4]:
def substituicao_regressiva(escalonada):
    # Resolve um sistema triangular superior com pivôs não nulos
    n = len(escalonada)
    x = [Fraction(0)] * n
    for i in range(n - 1, -1, -1):
        soma = sum(escalonada[i][k] * x[k] for k in range(i + 1, n))
        x[i] = (escalonada[i][n] - soma) / escalonada[i][i]
    return x


x_atual = substituicao_regressiva(escalonada)
x_numpy = np.linalg.solve(np.array(A, dtype=float), np.array(b, dtype=float))

print(f"{'Região':<8}{'Escalonamento':>15}{'NumPy':>10}")
for regiao, exato, aproximado in zip(REGIOES, x_atual, x_numpy):
    print(f"{regiao:<8}{str(exato):>15}{aproximado:>10.4f}")
print(f"{'Total':<8}{str(sum(x_atual)):>15}{x_numpy.sum():>10.4f}")

Região    Escalonamento     NumPy
Norte                12   12.0000
Sul                  20   20.0000
Leste                16   16.0000
Oeste                24   24.0000
Total                72   72.0000


**Distribuição atual recuperada:** $x = (N, S, L, O) = (12,\ 20,\ 16,\ 24)$.

Os dois métodos concordam. O NumPy trabalha com números de ponto flutuante, e por isso o escalonamento em frações é usado como resultado oficial: ele mostra que a resposta é **exatamente** inteira, sem depender de arredondamento.

### f. Verificação

Substituindo $x$ em cada registro do Relatório I:

In [5]:
indicadores = [
    "Frota total N + S + L + O",
    "N + 2S + O",
    "S + 2L",
    "N + L + 2O",
]
tudo_certo = True
for nome, linha, esperado in zip(indicadores, A, b):
    obtido = sum(coef * valor for coef, valor in zip(linha, x_atual))
    situacao = "ok" if obtido == esperado else "ERRO"
    tudo_certo &= obtido == esperado
    print(f"{nome:<28} = {str(obtido):>3}   (registro: {esperado})  {situacao}")

inteira = all(valor.denominator == 1 for valor in x_atual)
nao_negativa = all(valor >= 0 for valor in x_atual)
print()
print("Satisfaz os 4 registros:", tudo_certo)
print("Quantidades inteiras:   ", inteira)
print("Quantidades >= 0:       ", nao_negativa)

x_atual = [int(valor) for valor in x_atual]

Frota total N + S + L + O    =  72   (registro: 72)  ok
N + 2S + O                   =  76   (registro: 76)  ok
S + 2L                       =  52   (registro: 52)  ok
N + L + 2O                   =  76   (registro: 76)  ok

Satisfaz os 4 registros: True
Quantidades inteiras:    True
Quantidades >= 0:        True


A distribuição $(12, 20, 16, 24)$ satisfaz **os quatro registros**, é **inteira** e **não negativa**. Portanto ela é a única distribuição real compatível com os dados preservados.

---
## Etapa 2: política de redistribuição

### a. Ônibus necessários em cada região

Cada passageiro precisa de uma vaga e cada ônibus atende 40 passageiros no período. A necessidade da região $i$ é o menor número inteiro de ônibus que cobre a demanda:

$$ n_i = \left\lceil \frac{\text{passageiros}_i}{40} \right\rceil $$

O arredondamento para cima garante que ninguém fique sem vaga quando a divisão não é exata.

In [6]:
passageiros = [560, 760, 720, 840]
CAPACIDADE = 40

necessidade = [-(-p // CAPACIDADE) for p in passageiros]   # divisão arredondada para cima

print(f"{'Região':<8}{'Passageiros':>12}{'Passag./40':>12}{'Ônibus necessários':>20}")
for regiao, p, n in zip(REGIOES, passageiros, necessidade):
    print(f"{regiao:<8}{p:>12}{p / CAPACIDADE:>12.1f}{n:>20}")
print(f"{'Total':<8}{sum(passageiros):>12}{'':>12}{sum(necessidade):>20}")
print()
print("Frota disponível:", sum(x_atual))

Região   Passageiros  Passag./40  Ônibus necessários
Norte            560        14.0                  14
Sul              760        19.0                  19
Leste            720        18.0                  18
Oeste            840        21.0                  21
Total           2880                              72

Frota disponível: 72


Todas as divisões são exatas e a necessidade total é **72 ônibus, exatamente o tamanho da frota**. Isso tem duas consequências:

- **é possível atender toda a demanda sem comprar ônibus**, só redistribuindo;
- **não há folga**: depois da redistribuição ideal, todo ônibus roda com 100% da capacidade no pico, e qualquer aumento de demanda já falta vaga.

### b. Desequilíbrios

In [7]:
saldo = [atual - necessaria for atual, necessaria in zip(x_atual, necessidade)]

print(f"{'Região':<8}{'Atual':>7}{'Necessário':>12}{'Saldo':>8}{'Ocupação no pico':>19}   Situação")
for regiao, atual, necessaria, s, p in zip(REGIOES, x_atual, necessidade, saldo, passageiros):
    ocupacao = p / (atual * CAPACIDADE)
    situacao = "sobra" if s > 0 else ("falta" if s < 0 else "equilibrada")
    print(f"{regiao:<8}{atual:>7}{necessaria:>12}{s:>+8}{ocupacao:>18.1%}   {situacao}")

sem_vaga = sum(max(0, p - atual * CAPACIDADE) for p, atual in zip(passageiros, x_atual))
print()
print("Soma dos saldos:", sum(saldo))
print("Passageiros sem vaga hoje:", sem_vaga)

Região    Atual  Necessário   Saldo   Ocupação no pico   Situação
Norte        12          14      -2            116.7%   falta
Sul          20          19      +1             95.0%   sobra
Leste        16          18      -2            112.5%   falta
Oeste        24          21      +3             87.5%   sobra

Soma dos saldos: 0
Passageiros sem vaga hoje: 160


| Região | Situação hoje | Efeito |
|---|---|---|
| **Norte** | faltam **2** ônibus (ocupação de 116,7%) | 80 passageiros sem vaga no pico |
| **Leste** | faltam **2** ônibus (ocupação de 112,5%) | 80 passageiros sem vaga no pico |
| **Sul** | sobra **1** ônibus (ocupação de 95%) | 1 ônibus roda com folga |
| **Oeste** | sobram **3** ônibus (ocupação de 87,5%) | 3 ônibus rodam com folga |

O problema não é falta de frota (os saldos somam zero), e sim **má distribuição**: 160 passageiros ficam sem vaga no Norte e no Leste enquanto 4 ônibus têm folga no Sul e no Oeste.

### c. Política proposta

**Transferências:**

| De | Para | Ônibus |
|---|---|---|
| Oeste | Norte | 2 |
| Oeste | Leste | 1 |
| Sul | Leste | 1 |

**Critérios adotados:**

1. **Só cede quem tem sobra e só recebe quem tem falta.** Nenhuma região que está equilibrada ou em falta perde ônibus, e nenhuma região que cede fica abaixo da própria necessidade.
2. **Prioridade para regiões vizinhas.** O Oeste atende primeiro o Norte e o Sul atende o Leste, o que encurta o deslocamento vazio do ônibus até o novo centro operacional.
3. **O Oeste, que tem a maior sobra, cede a maior parte.** Depois de cobrir o Norte, o terceiro ônibus que sobra no Oeste completa o Leste junto com o ônibus do Sul.
4. **Mexer no mínimo de ônibus possível.** São movidos exatamente 4 ônibus, que é o total das faltas; não existe redistribuição que zere as faltas movendo menos.

### d. Representação como transformação linear

A política é escrita como uma transformação $T: \mathbb{R}^4 \to \mathbb{R}^4$, $T(x) = Mx$, em que cada região **envia uma fração fixa da sua frota** para cada destino:

| Origem | Frota | Envia | Fração enviada | Fração que fica |
|---|---|---|---|---|
| Norte | 12 | nada | 0 | 1 |
| Sul | 20 | 1 para o Leste | $1/20$ | $19/20$ |
| Leste | 16 | nada | 0 | 1 |
| Oeste | 24 | 2 para o Norte e 1 para o Leste | $2/24 = 1/12$ e $1/24$ | $21/24 = 7/8$ |

Assim, a nova frota de cada região é uma **combinação linear** das frotas atuais:

$$
\begin{aligned}
N' &= N + \tfrac{1}{12}\,O \\
S' &= \tfrac{19}{20}\,S \\
L' &= \tfrac{1}{20}\,S + L + \tfrac{1}{24}\,O \\
O' &= \tfrac{7}{8}\,O
\end{aligned}
$$

$T$ é linear porque é um produto por matriz: $T(\alpha x + \beta y) = M(\alpha x + \beta y) = \alpha Mx + \beta My = \alpha T(x) + \beta T(y)$.

### e. Matriz da transformação

$$
M = \begin{bmatrix}
1 & 0 & 0 & \tfrac{1}{12} \\
0 & \tfrac{19}{20} & 0 & 0 \\
0 & \tfrac{1}{20} & 1 & \tfrac{1}{24} \\
0 & 0 & 0 & \tfrac{7}{8}
\end{bmatrix}
\qquad
\begin{array}{l}
\text{colunas = região de origem (N, S, L, O)} \\
\text{linhas = região de destino (N, S, L, O)}
\end{array}
$$

**Significado dos coeficientes:** $m_{ij}$ é a **fração dos ônibus da região $j$ que passa a atender a região $i$**.

- **Diagonal** ($m_{ii}$): fração que **fica** na própria região. Norte e Leste ficam com tudo ($m_{11} = m_{33} = 1$); o Sul fica com $19/20$; o Oeste fica com $7/8$.
- **Fora da diagonal**: fração **transferida**. $m_{14} = 1/12$ é a parte do Oeste que vai para o Norte; $m_{34} = 1/24$ é a parte do Oeste que vai para o Leste; $m_{32} = 1/20$ é a parte do Sul que vai para o Leste.
- **Zeros**: rotas que a política não usa (por exemplo, $m_{21} = 0$: o Norte não envia ônibus para o Sul).
- **Cada coluna soma 1**: tudo o que sai de uma região chega em alguma região. Nenhum ônibus some e nenhum é criado.

In [8]:
M_ = fracoes([
    #  N   S                 L   O
    [1, 0,                0, Fraction(1, 12)],   # Norte recebe
    [0, Fraction(19, 20), 0, 0              ],   # Sul
    [0, Fraction(1, 20),  1, Fraction(1, 24)],   # Leste recebe
    [0, 0,                0, Fraction(7, 8) ],   # Oeste
])

def aplicar(matriz, vetor):
    return [sum(m * v for m, v in zip(linha, vetor)) for linha in matriz]

x_novo = aplicar(M_, x_atual)

mostrar_matriz(M_, "M =")
print()
print(f"{'Região':<8}{'Antes':>7}{'Depois':>8}{'Necessário':>12}{'Ocupação depois':>18}")
for regiao, antes, depois, necessaria, p in zip(REGIOES, x_atual, x_novo, necessidade, passageiros):
    print(f"{regiao:<8}{antes:>7}{str(depois):>8}{necessaria:>12}{p / (depois * CAPACIDADE):>18.1%}")
print(f"{'Total':<8}{sum(x_atual):>7}{str(sum(x_novo)):>8}{sum(necessidade):>12}")
print()
print("Atende exatamente a necessidade:", x_novo == necessidade)
print("Conferência com NumPy:", np.array(M_, dtype=float) @ np.array(x_atual, dtype=float))

M =
  [     1      0      0   1/12 ]
  [     0  19/20      0      0 ]
  [     0   1/20      1   1/24 ]
  [     0      0      0    7/8 ]

Região    Antes  Depois  Necessário   Ocupação depois
Norte        12      14          14            100.0%
Sul          20      19          19            100.0%
Leste        16      18          18            100.0%
Oeste        24      21          21            100.0%
Total        72      72          72

Atende exatamente a necessidade: True
Conferência com NumPy: [14. 19. 18. 21.]


### f. Demonstrações

**1) A transformação preserva o total de ônibus.**

Seja $\mathbf{1} = (1, 1, 1, 1)^T$. O total de uma distribuição é $\mathbf{1}^T x = N + S + L + O$. A soma de cada coluna de $M$ é 1:

$$
\text{col. N: } 1 = 1 \qquad
\text{col. S: } \tfrac{19}{20} + \tfrac{1}{20} = 1 \qquad
\text{col. L: } 1 = 1 \qquad
\text{col. O: } \tfrac{1}{12} + \tfrac{1}{24} + \tfrac{7}{8} = \tfrac{2 + 1 + 21}{24} = 1
$$

Isso é o mesmo que $\mathbf{1}^T M = \mathbf{1}^T$. Então, para **qualquer** distribuição $x$:

$$ \mathbf{1}^T T(x) = \mathbf{1}^T (Mx) = (\mathbf{1}^T M)\,x = \mathbf{1}^T x $$

ou seja, o total depois da transformação é igual ao total antes. Para a frota atual: $14 + 19 + 18 + 21 = 72 = 12 + 20 + 16 + 24$.

**2) A transformação não produz alocações negativas.**

Todos os coeficientes de $M$ são **maiores ou iguais a zero**, e uma distribuição válida tem $x_j \ge 0$. Cada nova quantidade é

$$ x'_i = \sum_{j=1}^{4} m_{ij}\, x_j $$

uma soma de produtos de números não negativos, logo $x'_i \ge 0$ para toda região $i$ e para **qualquer** distribuição $x \ge 0$.

Uma matriz com essas duas propriedades (entradas $\ge 0$ e colunas somando 1) é chamada de **matriz estocástica por colunas**, e as duas garantias valem para ela em geral, não só para os números deste caso.

A célula abaixo confere as duas propriedades na matriz e testa a transformação em 10.000 frotas aleatórias (isso não substitui a prova, só ilustra).

In [9]:
somas_colunas = [sum(M_[i][j] for i in range(4)) for j in range(4)]
menor_coeficiente = min(valor for linha in M_ for valor in linha)

print("Soma de cada coluna:", [str(s) for s in somas_colunas])
print("Todas as colunas somam 1:", all(s == 1 for s in somas_colunas))
print("Menor coeficiente de M:", menor_coeficiente, "-> nenhum negativo:", menor_coeficiente >= 0)

gerador = np.random.default_rng(2026)
M_float = np.array(M_, dtype=float)
total_ok = negativo = 0
for _ in range(10_000):
    frota = gerador.integers(0, 200, size=4)
    nova = M_float @ frota
    total_ok += np.isclose(nova.sum(), frota.sum())
    negativo += (nova < -1e-12).any()
print()
print(f"Total preservado em {total_ok} de 10.000 frotas aleatórias")
print(f"Frotas com alguma região negativa: {negativo}")

Soma de cada coluna: ['1', '1', '1', '1']
Todas as colunas somam 1: True
Menor coeficiente de M: 0 -> nenhum negativo: True


Total preservado em 10000 de 10.000 frotas aleatórias
Frotas com alguma região negativa: 0


### Avaliação do resultado

| Indicador | Antes | Depois |
|---|---|---|
| Distribuição (N, S, L, O) | (12, 20, 16, 24) | (14, 19, 18, 21) |
| Passageiros sem vaga no pico | 160 | 0 |
| Pior ocupação (região mais lotada) | 116,7% (Norte) | 100% (todas) |
| Ônibus com folga no pico | 4 | 0 |
| Ônibus movidos | | 4 |
| Frota total | 72 | 72 |

A redistribuição **zera a falta de vagas no pico** sem comprar nenhum ônibus.

### Limitações do modelo

- **Inteiros só neste caso.** A matriz usa frações da frota atual. Aplicada a outra frota, ela pode gerar quantidades fracionárias (por exemplo, $\tfrac{1}{12}$ de 30 ônibus = 2,5). Em uso real, seria preciso recalcular as frações para a nova frota ou arredondar mantendo o total.
- **Aplicar uma única vez.** $M$ descreve uma redistribuição. Aplicá-la de novo ($M^2x$) tiraria mais ônibus do Sul e do Oeste, que já estariam equilibrados.
- **Frota sem folga.** A necessidade total é igual à frota. Qualquer ônibus em manutenção ou aumento de demanda volta a deixar passageiros sem vaga; o modelo indica que **a frota está no limite**.
- **Dados de um único horário.** A pesquisa é do pico. Fora dele, a distribuição ideal pode ser outra. Com os contadores de pessoas do Ponto Vivo, a demanda poderia ser medida em vários horários e a matriz recalculada para cada um.

### Conclusão

Os quatro indicadores do Relatório I formam um sistema linear com $\det(A) = 3 \neq 0$; por isso existe **uma única** distribuição compatível, $(12, 20, 16, 24)$, que é inteira, não negativa e satisfaz todos os registros. Comparando com a demanda de pico, a frota total de 72 ônibus é exatamente a necessária, mas está mal distribuída: Norte e Leste ficam com 160 passageiros sem vaga, enquanto Sul e Oeste têm 4 ônibus com folga. A política proposta move 4 ônibus para regiões vizinhas e foi escrita como a transformação linear $T(x) = Mx$, com $M$ estocástica por colunas, o que garante matematicamente que o total é preservado e que nenhuma região fica negativa. O resultado, $(14, 19, 18, 21)$, atende 100% da demanda de pico.